# 3.0 — Théorie de l'information : entropie, KL, cross-entropy

**Navigation** : [<< 2.3-Régression linéaire et logistique](../02-ML-Cours/2.3-Regression-lineaire-logistique.ipynb) · [2.8-Théorie-PAC](../02-ML-Cours/2.8-Theorie-PAC.ipynb) · [2.9-Grokking](../02-ML-Cours/2.9-Grokking-Generalisation.ipynb) · [Feuille de route de la série](README.md) · [Version Python](3.0-Theorie-Information.ipynb) · [3.1 — La rétropropagation >>](3.1-Retropropagation.ipynb)

Le notebook [2.3](../02-ML-Cours/2.3-Regression-lineaire-logistique.ipynb) vous a montré que la régression logistique est ajustée par le **maximum de vraisemblance**. Ce notebook ouvre la boîte de ce qu'on optimise vraiment : la **cross-entropy**. C'est la première pierre de la série 03 — avant d'écrire la rétropropagation (3.1) et les optimiseurs (3.2), il faut savoir *ce que* la loss mesure.

Toute la théorie de l'information tient en trois quantités. On les construit ici **from scratch**, on les **vérifie** numériquement sur un vrai texte français, puis on les relie aux losses que le reste de la série consomme : le `LogSoftmax` d'une tête de classification, le penalty KL du DPO ([PT_03](../../../GenAI/PostTraining/PT_03_dpo_direct_preference.ipynb)) et le ratio KL de la régularisation PPO/GRPO ([PT_04](../../../GenAI/PostTraining/PT_04_grpo_deepseek_r1.ipynb)).

**Objectifs** : calculer H, CE et KL, comparer deux objectifs de classification et mesurer l’effet de la température.

**Durée conseillée** : 45 minutes.

**Kernel** : .NET C# · **Bibliothèques** : Python.NET et scikit-learn pour les données, BCL pour les calculs · **CPU** : oui (~1 min) · **Prérequis** : 2.3 (régression logistique), 2.8 (bornes), un peu de probabilités


***

## 1. L'entropie : la surprise moyenne d'une source

Un **message** est un tirage de caractères selon une distribution $p$. L'information apportée par un symbole $x$ est sa **surprise** : $-\log_2 p(x)$, en bits. Un symbole fréquent ($p$ grand) surprend peu, un symbole rare ($p$ petit) surprend beaucoup. L'**entropie** $H(p) = -\sum_x p(x) \log_2 p(x)$ est l'information moyenne par tirage — le taux idéal en bits par symbole ; un code préfixe symbole par symbole impose des longueurs entières, tandis que le codage par blocs peut approcher ce taux.

Deux bornes la cadrent : $H(p) \ge 0$ (la surprise n'est pas négative) et $H(p) \le \log_2 K$ où $K$ est la taille de l'alphabet (l'entropie est maximale quand la distribution est **uniforme**). Le rapport $H / \log_2 K$ mesure l'**efficacité** de la source ; son complément $1 - H / \log_2 K$ est la **redondance**. Plus le rapport est bas, plus la source est compressible.

On mesure tout ça sur un vrai texte français (une fable de La Fontaine, domaine public), au niveau des caractères.


In [1]:
#r "nuget: PythonNet, 3.0.5"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
using System.Text;
using Python.Runtime;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
string texte = "Maître Corbeau, sur un arbre perché, tenait en son bec un fromage.\nMaître Renard, par l'odeur alléché, lui tint à peu près ce langage :\n« Bonjour, Monsieur du Corbeau. Que vous êtes joli ! que vous me semblez beau !\nSans mentir, si votre ramage se rapporte à votre plumage, vous êtes le Phénix\ndes hôtes de ces bois. » À ces mots le Corbeau ne se sent pas de joie ; et pour\nmontrer sa belle voix, il ouvre un large bec, laisse tomber sa proie. Le Renard\ns'en saisit, et dit : « Mon bon Monsieur, apprenez que tout flatteur vit aux\ndépens de celui qui l'écoute : cette leçon vaut bien un fromage, sans doute. »\nLe Corbeau, honteux et confus, jura, mais un peu tard, qu'on ne l'y prendrait plus.\n\nLa Cigale, ayant chanté tout l'été, se trouva fort dépourvue quand la bise fut\nvenue : pas un seul petit morceau de mouche ou de vermisseau. Elle alla crier\nfamine chez la Fourmi sa voisine, la priant de lui prêter quelque grain pour\nsubsister jusqu'à la saison nouvelle. « Je vous paierai, lui dit-elle, avant\nl'août, foi d'animal, intérêt et principal. » La Fourmi n'est pas prêteuse :\nc'est là son moindre défaut. « Que faisiez-vous au temps chaud ? dit-elle à\ncette emprunteuse. — Nuit et jour à tout venant je chantais, ne vous déplaise.\n— Vous chantiez ? j'en suis fort aise. Eh bien ! dansez maintenant. »";
var alphabet = texte.Distinct().OrderBy(c => c).ToArray();
var occ = alphabet.Select(c => texte.Count(x => x == c)).ToArray();
var p = occ.Select(n => (double)n / texte.Length).ToArray();
double Ent(double[] a)
{
    if (a.Length == 0 || a.Any(x => !double.IsFinite(x) || x < 0) || Math.Abs(a.Sum() - 1) > 1e-12)
        throw new ArgumentException("distribution invalide");
    return -a.Where(x => x > 0).Sum(x => x * Math.Log2(x));
}

Console.WriteLine($".NET {Environment.Version} ; Python.NET {typeof(PythonEngine).Assembly.GetName().Version}");
Console.WriteLine($"caractères {texte.Length}, symboles {alphabet.Length}, H={Ent(p):F4} bits/caractère, Hmax={Math.Log2(alphabet.Length):F4}, redondance={1 - Ent(p) / Math.Log2(alphabet.Length):P1}");


Installed Packages PythonNet, 3.0.5

.NET 10.0.14 ; Python.NET 3.0.5.0


caractères 1307, symboles 59, H=4.5334 bits/caractère, Hmax=5.8826, redondance=22.9 %


**Lecture.** Sur ces 1 307 caractères et 59 symboles, l'entropie empirique vaut 4,5334 bits/caractère contre 5,8826 bits au maximum uniforme, soit 22,9 % de redondance. Ce seul texte ne définit pas une loi universelle du français ; cette statistique unigramme ne mesure pas les dépendances entre caractères.

## Exercice 1 — Entropie du second texte

Implémentez l'entropie en bits par caractère et comparez le texte français au texte anglais fourni, inchangé. Expliquez pourquoi la redondance normalisée d'un pangramme court est biaisée. Refusez un texte vide. Le contrôle compare plusieurs textes non triviaux.

In [2]:
string texte2 = "The quick brown fox jumps over the lazy dog. Pack my box with five dozen liquor jugs.";
double? EntropieTexte(string s)
{ /* TODO étudiant */
    return null;
}

bool VerifierEntropie(Func<string, double?> f)
{
    foreach (var s in new[]
    {
        "aaaa",
        "abab",
        "abbc",
        texte2
    }

    )
    {
        var v = f(s);
        if (v is null)
            return false;
        var attendu = -s.GroupBy(c => c).Sum(g =>
        {
            double p = (double)g.Count() / s.Length;
            return p * Math.Log2(p);
        });
        if (!double.IsFinite(v.Value) || Math.Abs(v.Value - attendu) > 1e-10)
            throw new Exception("entropie incorrecte");
    }

    try
    {
        f("");
        throw new Exception("texte vide accepté");
    }
    catch (ArgumentException)
    {
    }

    return true;
}

Console.WriteLine(VerifierEntropie(EntropieTexte) ? "Exercice 1 validé" : "Exercice 1 à compléter");


Exercice 1 à compléter


***

## 2. Cross-entropy et divergence de KL : le coût de croire un modèle

L'entropie $H(p)$ suppose que *je connais* la vraie distribution $p$. En apprentissage, on ne la connaît pas : on a un **modèle** qui produit $q$, et on veut mesurer combien il coûte d'utiliser $q$ alors que la vérité est $p$. La **cross-entropy** $H(p, q) = -\sum_x p(x) \log_2 q(x)$ est ce coût : le coût idéal d’un code ajusté à $q$, évalué sur des tirages de $p$.

La **divergence de Kullback–Leibler** $D_{KL}(p \| q) = \sum_x p(x) \log_2 \frac{p(x)}{q(x)}$ isole le **surcoût** de $q$ par rapport à $p$ : c'est la « distance » que $q$ ajoute, en bits, au prix optimal. Les deux sont liées par l'identité fondamentale $H(p, q) = H(p) + D_{KL}(p \| q)$.

Deux propriétés font de la KL un outil de théorie de l'apprentissage : la **positivité** $D_{KL}(p \| q) \ge 0$ (inégalité de Gibbs, égalité ssi $p = q$), et son **asymétrie** — $D_{KL}(p \| q) \neq D_{KL}(q \| p)$ — ce qui en fait une *divergence*, pas une métrique. On vérifie les deux numériquement, avec le piège classique du $\log 0$.


In [3]:
double[] Norm(double[] v)
{
    if (v.Length == 0 || v.Any(x => !double.IsFinite(x) || x < 0) || v.Max() <= 0)
        throw new ArgumentException("poids invalides");
    double max = v.Max();
    var scaled = v.Select(x => x / max).ToArray();
    double sum = scaled.Sum();
    return scaled.Select(x => x / sum).ToArray();
}

void Check(double[] a, double[] b)
{
    if (a.Length == 0 || a.Length != b.Length || a.Any(x => !double.IsFinite(x) || x < 0) || b.Any(x => !double.IsFinite(x) || x < 0) || Math.Abs(a.Sum() - 1) > 1e-12 || Math.Abs(b.Sum() - 1) > 1e-12)
        throw new ArgumentException("distributions invalides");
}

double CE(double[] a, double[] b)
{
    Check(a, b);
    double s = 0;
    for (int i = 0; i < a.Length; i++)
        if (a[i] > 0)
        {
            if (b[i] == 0)
                return double.PositiveInfinity;
            s -= a[i] * Math.Log2(b[i]);
        }

    return s;
}

double KL(double[] a, double[] b)
{
    Check(a, b);
    double s = 0;
    for (int i = 0; i < a.Length; i++)
        if (a[i] > 0)
        {
            if (b[i] == 0)
                return double.PositiveInfinity;
            s += a[i] * (Math.Log2(a[i]) - Math.Log2(b[i]));
        }

    return s;
}

var rng = new Random(7);
var q = Norm(p.Select(x => Math.Pow(x, 1.3) + .02 * rng.NextDouble()).ToArray());
var min = double.PositiveInfinity;
for (int i = 0; i < 21; i++)
    for (int j = 0; j < 4; j++)
    {
        double a = .5 + i * .1;
        var z = Norm(p.Select(x => Math.Pow(x, a) + .01 * j).ToArray());
        min = Math.Min(min, KL(p, z));
    }

var qzero = (double[])q.Clone();
qzero[Array.IndexOf(p, p.Max())] = 0;
qzero = Norm(qzero);
if (CE(new[] { 1d, 0 }, new[] { 1d, 0 }) != 0 || !double.IsPositiveInfinity(CE(new[] { 1d, 0 }, new[] { 0d, 1 })))
    throw new Exception("zéros");
Console.WriteLine($"H={Ent(p):F6}, CE={CE(p, q):F6}, KL={KL(p, q):F6} bits, identité écart {Math.Abs(CE(p, q) - Ent(p) - KL(p, q)):E2}, KL inverse {KL(q, p):F6}, minimum balayage {min:E2}");
Console.WriteLine($"CE zéro={CE(p, qzero)} ; lissage normalisé alpha=1e-12 : {CE(p, Norm(qzero.Select(x => x + 1e-12).ToArray())):F4} bits");
foreach (var bad in new[]
{
    Array.Empty<double>(),
    new[]
    {
        0d,
        0d
    },
    new[]
    {
        double.NaN,
        1d
    },
    new[]
    {
        double.PositiveInfinity,
        0d
    },
    new[]
    {
        -1d,
        2d
    },
    new[]
    {
        .2,
        .2
    }
}

)
{
    try
    {
        CE(bad, bad);
        throw new Exception("distribution invalide acceptée");
    }
    catch (ArgumentException)
    {
    }
}

Console.WriteLine("Contrats distributions invalides : OK");
var large = Norm(new[] { 1e308, 1e308 });
if (Math.Abs(large[0] - .5) > 1e-15 || Math.Abs(KL(new[] { 1d, 0 }, new[] { 1e-320, 1d }) + Math.Log2(1e-320)) > 1e-10)
    throw new Exception("poids extrêmes ou KL subnormale");
Console.WriteLine("Poids extrêmes et KL subnormale : OK");


H=4.533427, CE=5.069030, KL=0.535603 bits, identité écart 1.11E-015, KL inverse 0.871174, minimum balayage -2.62E-016


CE zéro=Infinity ; lissage normalisé alpha=1e-12 : 11.3355 bits


Contrats distributions invalides : OK


Poids extrêmes et KL subnormale : OK


**Lecture.** La CE est l'entropie plus la KL, non négative et asymétrique. Le générateur C# diffère de NumPy ; la perturbation et les valeurs exactes diffèrent. Un zéro de q sur le support de p donne une CE infinie ; le lissage additif normalisé fournit une nouvelle distribution et une CE finie.

## Exercice 2 — Lissage normalisé

Pour une distribution p et des poids q de mêmes dimensions, ajoutez alpha à chaque poids de q, normalisez puis calculez la cross-entropy en bits. Alpha doit être fini et strictement positif : montrez que la perte est finie même si q s'annule sur le support de p, sans supposer une variation monotone avec alpha. Refusez dimensions différentes, vecteurs vides, valeurs non finies ou négatives et p de masse autre que 1.

In [4]:
double? CeLisse(double[] a, double[] b, double alpha)
{ /* TODO étudiant */
    return null;
}

bool VerifierLissage(Func<double[], double[], double, double?> f)
{
    foreach (var test in new[]
    {
        (new[] { 1d, 0 }, new[] { 0d, 1 }, .5),
        (new[] { .25, .75 }, new[] { 2d, 1 }, .1),
        (new[] { 0d, 1 }, new[] { 1d, 0 }, 2d)
    }

    )
    {
        var(a, b, alpha) = test;
        var val = f(a, b, alpha);
        if (val is null)
            return false;
        var smooth = b.Select(x => (x + alpha) / (b.Sum() + b.Length * alpha)).ToArray();
        double attendu = -a.Zip(smooth, (x, z) => x == 0 ? 0 : x * Math.Log2(z)).Sum();
        if (!double.IsFinite(val.Value) || Math.Abs(val.Value - attendu) > 1e-10)
            throw new Exception("lissage incorrect");
    }

    foreach (var t in new[]
    {
        (new[] { 1d, 0 }, new[] { 1d }, .1),
        (new[] { 1d, 0 }, new[] { 0d, 1 }, 0d),
        (new[] { 1d, 0 }, new[] { 0d, 1 }, double.NaN),
        (new[] { 1d, 0 }, new[] { double.NaN, 1d }, .1),
        (new[] { 1d, 0 }, new[] { double.PositiveInfinity, 1d }, .1),
        (new[] { double.NaN, 0d }, new[] { 1d, 0d }, .1),
        (new[] { .2, .2 }, new[] { 1d, 0d }, .1),
        (new[] { 1d, 0 }, new[] { 0d, 1d }, double.PositiveInfinity),
        (new[] { 1d, 0 }, new[] { 0d, 1d }, -1d)
    }

    )
    {
        try
        {
            f(t.Item1, t.Item2, t.Item3);
            throw new Exception("entrée invalide acceptée");
        }
        catch (ArgumentException)
        {
        }
    }

    return true;
}

Console.WriteLine(VerifierLissage(CeLisse) ? "Exercice 2 validé" : "Exercice 2 à compléter");


Exercice 2 à compléter


***

## 3. MSE vs cross-entropy : effet de la saturation sur les gradients

Une régression linéaire (2.3) minimise les **moindres carrés** (MSE). Une régression logistique minimise la **cross-entropy** (maximum de vraisemblance). Les deux peuvent techniquement s'appliquer à un classifieur — mais elles ne se comportent pas du tout pareil. On le montre ici littéralement : le même modèle, écrit from scratch, entraîné avec $L = \tfrac12\mathrm{MSE}$ puis avec $L = H(p, q)$, et on mesure la différence.

Le mécanisme est un **gradient qui disparaît**. Pour la sortie sigmoïde $\sigma(z)$, en notant $p = \sigma(z)$ :

- **Demi-MSE** : le gradient passe par $p(1-p)$, la dérivée de la sigmoïde. Quand le modèle est **saturé et faux** ($p \to 0$ alors que $y = 1$), ce facteur $p(1-p) \to 0$ — le gradient meurt, le modèle reste faux.
- **Cross-entropy** : le gradient passe par $(p - y)$ directement. Même saturé, un exemple mal classé garde un gradient de l'ordre de $1$ — le modèle continue d'apprendre.

Le premier terme $(p - y)$ est le même dans les deux cas ; c'est le facteur $p(1-p)$ qui les sépare.


In [5]:
if (!PythonEngine.IsInitialized)
{
    Runtime.PythonDLL = Environment.GetEnvironmentVariable("PYTHONNET_PYDLL");
    PythonEngine.Initialize();
    PythonEngine.BeginAllowThreads();
}

double[][] X = null;
double[] y = null;
using (Py.GIL())
{
    dynamic ds = Py.Import("sklearn.datasets"), sys = Py.Import("sys"), sk = Py.Import("sklearn");
    using PyObject data = ds.make_classification(n_samples: 400, n_features: 2, n_redundant: 0, n_clusters_per_class: 1, class_sep: 1.0, random_state: 42);
    using PyObject xx = data.GetItem(0);
    using PyObject yy = data.GetItem(1);
    double[][] rawX = (double[][])((dynamic)xx).tolist().As<double[][]>();
    long[] rawY = (long[])((dynamic)yy).tolist().As<long[]>();
    X = rawX.Select(a => new[] { a[0], a[1], 1.0 }).ToArray();
    y = rawY.Select(v => (double)v).ToArray();
    Console.WriteLine($"CPython {sys.version} ; sklearn {sk.__version__} ; dataset {X.Length}×2, classes {y.Count(v => v == 0)}/{y.Count(v => v == 1)}");
}

double Sig(double z) => z >= 0 ? 1 / (1 + Math.Exp(-z)) : Math.Exp(z) / (1 + Math.Exp(z));
double Soft(double z) => Math.Max(0, z) + Math.Log(1 + Math.Exp(-Math.Abs(z)));
(double[], double[]) Train(double[] start, bool ce, double lr, int steps)
{
    var w = (double[])start.Clone();
    var hist = new double[steps];
    for (int k = 0; k < steps; k++)
    {
        var g = new double[3];
        double loss = 0;
        for (int i = 0; i < y.Length; i++)
        {
            double z = X[i].Zip(w, (a, b) => a * b).Sum(), pred = Sig(z), err = pred - y[i];
            loss += ce ? (y[i] == 1 ? Soft(-z) : Soft(z)) : .5 * err * err;
            double d = ce ? err : err * pred * (1 - pred);
            for (int j = 0; j < 3; j++)
                g[j] += d * X[i][j];
        }

        hist[k] = loss / y.Length;
        for (int j = 0; j < 3; j++)
            w[j] -= lr * g[j] / y.Length;
    }

    return (w, hist);
}

double Acc(double[] w) => Enumerable.Range(0, y.Length).Count(i => (Sig(X[i].Zip(w, (a, b) => a * b).Sum()) >= .5) == (y[i] == 1)) / (double)y.Length;
var(w0, h0) = Train(new double[3], true, .5, 800);
Console.WriteLine($"CE depuis zéro 800 pas : perte {h0.Last():F4}, accuracy {Acc(w0):F3}");
// Contrôles numériques indépendants des deux dérivées et des logits extrêmes.
foreach (double z in new[]
{
    -1000d,
    1000d
}

)
    foreach (double label in new[]
    {
        0d,
        1d
    }

    )
        if (!double.IsFinite(label == 1 ? Soft(-z) : Soft(z)) || !double.IsFinite(Sig(z)))
            throw new Exception("logit extrême");
foreach (bool ce in new[]
{
    true,
    false
}

)
{
    double z = .37, label = 1, delta = 1e-5;
    Func<double, double> loss = t => ce ? Soft(-t) : .5 * Math.Pow(Sig(t) - label, 2);
    double fd = (loss(z + delta) - loss(z - delta)) / (2 * delta);
    double pred = Sig(z), grad = ce ? pred - label : (pred - label) * pred * (1 - pred);
    if (Math.Abs(fd - grad) > 1e-9)
        throw new Exception("gradient");
}

Console.WriteLine("Contrôles logits extrêmes et différences finies : OK");


CPython 3.13.16 (main, Oct 14 2026, 01:04:01) [Clang 23.1.3 ] ; sklearn 1.9.1 ; dataset 400×2, classes 200/200


CE depuis zéro 800 pas : perte 0.2942, accuracy 0.887


Contrôles logits extrêmes et différences finies : OK


Comparons maintenant les deux pertes depuis la même initialisation saturée et visualisons la frontière CE.

In [6]:
using Microsoft.DotNet.Interactive.Formatting;
using System.Net;

record Svg(string Content);
Formatter.Register<Svg>((v, w) => w.Write(v.Content), "text/html");
string F(double x) => x.ToString("F2", CultureInfo.InvariantCulture);
string Esc(string x) => WebUtility.HtmlEncode(x);
var start = new[]
{
    -8d,
    -8d,
    -8d
};
var(wc, hc) = Train(start, true, .2, 400);
var(wm, hm) = Train(start, false, .2, 400);
Console.WriteLine($"Accuracy initiale {Acc(start):F3} ; CE {Acc(wc):F3} ; demi-MSE {Acc(wm):F3} ; rapport gradients {1 / (.02 * .98):F1}");
var svg = new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='1000' height='420' viewBox='0 0 1000 420'><rect width='1000' height='420' fill='white'/>");
void Label(double x, double y, string s) => svg.Append($"<text x='{F(x)}' y='{F(y)}' font-size='13'>{Esc(s)}</text>");
void Line(double x1, double y1, double x2, double y2, string color = "black") => svg.Append($"<line x1='{F(x1)}' y1='{F(y1)}' x2='{F(x2)}' y2='{F(y2)}' stroke='{color}'/>");
Line(60, 350, 465, 350);
Line(60, 50, 60, 350);
double ymax = Math.Max(hc.Max(), hm.Max());
for (int i = 0; i <= 4; i++)
{
    double x = 60 + i * 100;
    Line(x, 350, x, 355);
    Label(x - 8, 373, $"{i * 100}");
    double v = ymax * i / 4;
    double yy = 350 - i * 75;
    Line(55, yy, 60, yy);
    Label(5, yy + 4, $"{v:F1}");
}

void Curve(double[] h, string color)
{
    svg.Append($"<polyline fill='none' stroke='{color}' stroke-width='2' points='");
    for (int i = 0; i < h.Length; i++)
        svg.Append($"{F(60 + i * 400.0 / 399)},{F(350 - 300 * h[i] / ymax)} ");
    svg.Append("'/>");
}

Curve(hc, "blue");
Curve(hm, "red");
Label(180, 400, "Itération");
Label(65, 30, "Perte : CE (bleu), demi-MSE (rouge)");
double xmin = X.Min(a => a[0]) - .2, xmax = X.Max(a => a[0]) + .2, ymin = X.Min(a => a[1]) - .2, ymax2 = X.Max(a => a[1]) + .2;
double Px(double v) => 550 + 400 * (v - xmin) / (xmax - xmin);
double Py(double v) => 350 - 300 * (v - ymin) / (ymax2 - ymin);
Line(550, 350, 950, 350);
Line(550, 50, 550, 350);
for (int i = 0; i <= 4; i++)
{
    double v = xmin + (xmax - xmin) * i / 4;
    Line(550 + i * 100, 350, 550 + i * 100, 355);
    Label(540 + i * 100, 373, $"{v:F1}");
    double u = ymin + (ymax2 - ymin) * i / 4;
    Line(545, 350 - i * 75, 550, 350 - i * 75);
    Label(500, 354 - i * 75, $"{u:F1}");
}

for (int i = 0; i < X.Length; i++)
    svg.Append($"<circle cx='{F(Px(X[i][0]))}' cy='{F(Py(X[i][1]))}' r='2' fill='{(y[i] == 0 ? "blue" : "red")}'/>");
svg.Append("<polyline fill='none' stroke='black' stroke-width='2' points='");
for (int i = 0; i < 200; i++)
{
    double x = xmin + (xmax - xmin) * i / 199;
    double v = -(wc[0] * x + wc[2]) / wc[1];
    if (v >= ymin && v <= ymax2)
        svg.Append($"{F(Px(x))},{F(Py(v))} ");
}

svg.Append("'/>");
Label(560, 30, "Classe 0 (bleu), classe 1 (rouge), frontière CE (noir)");
Label(735, 400, "x1");
Label(520, 30, "x2");
svg.Append("</svg>");
display(new Svg(svg.ToString()));


Accuracy initiale 0.375 ; CE 0.882 ; demi-MSE 0.393 ; rapport gradients 51.0


0 0.0 100 3.3 200 6.5 300 9.8 400 13.1 <polyline fill='none' stroke='blue' stroke-width='2' points='60.00,50.00 61.00,54.51 62.01,59.01 63.01,63.50 64.01,67.98 65.01,72.46 66.02,76.92 67.02,81.37 68.02,85.81 69.02,90.24 70.03,94.65 71.03,99.06 72.03,103.45 73.03,107.84 74.04,112.20 75.04,116.56 76.04,120.90 77.04,125.23 78.05,129.54 79.05,133.84 80.05,138.12 81.05,142.39 82.06,146.64 83.06,150.87 84.06,155.09 85.06,159.28 86.07,163.46 87.07,167.62 88.07,171.76 89.07,175.88 90.08,179.98 91.08,184.06 92.08,188.11 93.08,192.15 94.09,196.17 95.09,200.16 96.09,204.13 97.09,208.09 98.10,212.02 99.10,215.93 100.10,219.82 101.10,223.69 102.11,227.55 103.11,231.38 104.11,235.20 105.11,239.00 106.12,242.78 107.12,246.54 108.12,250.29 109.12,254.03 110.13,257.74 111.13,261.45 112.13,265.13 113.13,268.80 114.14,272.45 115.14,276.08 116.14,279.70 117.14,283.28 118.15,286.84 119.15,290.37 120.15,293.85 121.15,297.28 122.16,300.64 123.16,303.92 124.16,307.10 125.16,310.17 126.17,313.11 127.17,315.90 128.17,318.53 129.17,320.98 130.18,323.26 131.18,325.36 132.18,327.27 133.18,329.01 134.19,330.58 135.19,331.99 136.19,333.25 137.19,334.38 138.20,335.38 139.20,336.27 140.20,337.06 141.20,337.75 142.21,338.37 143.21,338.91 144.21,339.39 145.21,339.81 146.22,340.17 147.22,340.50 148.22,340.78 149.22,341.03 150.23,341.25 151.23,341.45 152.23,341.62 153.23,341.77 154.24,341.90 155.24,342.01 156.24,342.12 157.24,342.20 158.25,342.28 159.25,342.35 160.25,342.41 161.25,342.47 162.26,342.52 163.26,342.56 164.26,342.60 165.26,342.63 166.27,342.66 167.27,342.68 168.27,342.71 169.27,342.73 170.28,342.75 171.28,342.76 172.28,342.78 173.28,342.79 174.29,342.80 175.29,342.81 176.29,342.82 177.29,342.83 178.30,342.84 179.30,342.85 180.30,342.86 181.30,342.86 182.31,342.87 183.31,342.87 184.31,342.88 185.31,342.88 186.32,342.89 187.32,342.89 188.32,342.90 189.32,342.90 190.33,342.90 191.33,342.91 192.33,342.91 193.33,342.91 194.34,342.91 195.34,342.92 196.34,342.92 197.34,342.92 198.35,342.92 199.35,342.93 200.35,342.93 201.35,342.93 202.36,342.93 203.36,342.93 204.36,342.94 205.36,342.94 206.37,342.94 207.37,342.94 208.37,342.94 209.37,342.94 210.38,342.95 211.38,342.95 212.38,342.95 213.38,342.95 214.39,342.95 215.39,342.95 216.39,342.96 217.39,342.96 218.40,342.96 219.40,342.96 220.40,342.96 221.40,342.96 222.41,342.96 223.41,342.96 224.41,342.97 225.41,342.97 226.42,342.97 227.42,342.97 228.42,342.97 229.42,342.97 230.43,342.97 231.43,342.98 232.43,342.98 233.43,342.98 234.44,342.98 235.44,342.98 236.44,342.98 237.44,342.98 238.45,342.98 239.45,342.98 240.45,342.99 241.45,342.99 242.46,342.99 243.46,342.99 244.46,342.99 245.46,342.99 246.47,342.99 247.47,342.99 248.47,342.99 249.47,342.99 250.48,343.00 251.48,343.00 252.48,343.00 253.48,343.00 254.49,343.00 255.49,343.00 256.49,343.00 257.49,343.00 258.50,343.00 259.50,343.00 260.50,343.01 261.50,343.01 262.51,343.01 263.51,343.01 264.51,343.01 265.51,343.01 266.52,343.01 267.52,343.01 268.52,343.01 269.52,343.01 270.53,343.01 271.53,343.02 272.53,343.02 273.53,343.02 274.54,343.02 275.54,343.02 276.54,343.02 277.54,343.02 278.55,343.02 279.55,343.02 280.55,343.02 281.55,343.02 282.56,343.02 283.56,343.03 284.56,343.03 285.56,343.03 286.57,343.03 287.57,343.03 288.57,343.03 289.57,343.03 290.58,343.03 291.58,343.03 292.58,343.03 293.58,343.03 294.59,343.03 295.59,343.03 296.59,343.04 297.59,343.04 298.60,343.04 299.60,343.04 300.60,343.04 301.60,343.04 302.61,343.04 303.61,343.04 304.61,343.04 305.61,343.04 306.62,343.04 307.62,343.04 308.62,343.04 309.62,343.04 310.63,343.04 311.63,343.05 312.63,343.05 313.63,343.05 314.64,343.05 315.64,343.05 316.64,343.05 317.64,343.05 318.65,343.05 319.65,343.05 320.65,343.05 321.65,343.05 322.66,343.05 323.66,343.05 324.66,343.05 325.66,343.05 326.67,343.06 327.67,343.06 328.67,343.06 329.67,343.06 330.68,343.06 331.68,343.06 332.68,343.06 333.68,343.06 334.69,343.06 335.69,343.06 336.69,343.06 337.69,343.06 338.70,343.06 339.70,343.06 340.70,343.06 341.70,343.06 342.

**Lecture.** Depuis l'initialisation saturée, l'accuracy passe de 0,375 à 0,882 avec CE et à 0,393 avec demi-MSE après 400 pas à 0,2. Pour p=0,02 et y=1 le gradient CE vaut 0,9800 en valeur absolue contre 0,01921 pour la demi-MSE : le facteur p(1-p) explique ce rapport de 51. La MSE reste un objectif possible, notamment depuis une autre initialisation ; cette expérience ne démontre pas une interdiction générale.

***

## 4. KL comme distance entre modèles : un mini-modèle de langue et la température

La section 2 a mesuré $D_{KL}(p \| q)$ entre la vérité $p$ et un modèle $q$. Ici on inverse le regard : comparer entre elles deux **distributions de modèle** — c'est une notion utilisée par la régularisation de la série PostTraining, où l'on veut mesurer combien une policy s'éloigne d'une référence.

On construit un **mini-modèle de langue** from scratch : un **bigramme** de caractères entraîné sur le texte français (la probabilité du prochain caractère conditionnée au précédent). C'est un modèle réaliste — non trivial, mais assez petit pour tourner sur CPU. Deux usages :

1. **Qualité prédictive** : un bigramme peut prédire mieux qu'un unigramme (il capte la structure des suites), donc sa cross-entropy en bits/caractère est plus basse.
2. **Distance par température** : le **sampling par température** applique des logits ré-échelonnés $\mathrm{softmax}(z / T)$ ; une température $T < 1$ **concentre** la distribution, $T > 1$ l'**aplatit**. La KL entre la distribution de référence ($T = 1$) et sa version chauffée/refroidie mesure ce décalage.


In [7]:
var idx = alphabet.Select((c, i) => (c, i)).ToDictionary(t => t.c, t => t.i);
int A = alphabet.Length;
var big = new double[A, A];
for (int i = 0; i < A; i++)
    for (int j = 0; j < A; j++)
        big[i, j] = 1;
for (int i = 1; i < texte.Length; i++)
    big[idx[texte[i - 1]], idx[texte[i]]]++;
for (int i = 0; i < A; i++)
{
    double sum = 0;
    for (int j = 0; j < A; j++)
        sum += big[i, j];
    for (int j = 0; j < A; j++)
        big[i, j] /= sum;
}

double uni = 0, bi = 0;
for (int i = 1; i < texte.Length; i++)
{
    uni -= Math.Log2(p[idx[texte[i]]]);
    bi -= Math.Log2(big[idx[texte[i - 1]], idx[texte[i]]]);
}

uni /= texte.Length - 1;
bi /= texte.Length - 1;
Console.WriteLine($"Transitions entraînement {texte.Length - 1} ; unigramme apparié {uni:F3}, bigramme {bi:F3}, gain {uni - bi:F3} bits/caractère ; unigramme complet {Ent(p):F3}");


Transitions entraînement 1306 ; unigramme apparié 4.531, bigramme 4.009, gain 0.522 bits/caractère ; unigramme complet 4.533


La distribution conditionnelle du contexte espace permet maintenant de mesurer la KL selon la température.

In [8]:
double[] Temp(double[] r, double t)
{
    if (!double.IsFinite(t) || t <= 0 || r.Length == 0 || r.Any(x => !double.IsFinite(x) || x <= 0) || Math.Abs(r.Sum() - 1) > 1e-12)
        throw new ArgumentException("température ou support invalide");
    var log = r.Select(x => Math.Log(x)).ToArray();
    double m = log.Max();
    return Norm(log.Select(x => Math.Exp((x - m) / t)).ToArray());
}

var reference = Enumerable.Range(0, A).Select(j => big[idx[' '], j]).ToArray();
foreach (var t in new[]
{
    .3,
    1d,
    2d
}

)
{
    var dist = Temp(reference, t);
    Console.WriteLine($"T={t:F1}, KL={KL(reference, dist):F4} bits, H={Ent(dist):F4}");
}

var ts = Enumerable.Range(0, 29).Select(i => .2 + i * .1).ToArray();
var ks = ts.Select(t => KL(reference, Temp(reference, t))).ToArray();
Console.WriteLine($"Minimum à T={ts[Array.IndexOf(ks, ks.Min())]:F1}");
var chart = new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='700' height='420' viewBox='0 0 700 420'><rect width='700' height='420' fill='white'/><line x1='70' y1='350' x2='650' y2='350' stroke='black'/><line x1='70' y1='50' x2='70' y2='350' stroke='black'/>");
double km = ks.Max();
for (int i = 0; i <= 4; i++)
{
    double x = 70 + 580 * i / 4, yv = 350 - 300 * i / 4;
    chart.Append($"<text x='{F(x - 10)}' y='374' font-size='12'>{F(.2 + 2.8 * i / 4)}</text><text x='10' y='{F(yv)}' font-size='12'>{F(km * i / 4)}</text><line x1='{F(x)}' y1='350' x2='{F(x)}' y2='355' stroke='black'/><line x1='65' y1='{F(yv)}' x2='70' y2='{F(yv)}' stroke='black'/>");
}

chart.Append($"<line x1='{F(70 + 580 * .8 / 2.8)}' y1='50' x2='{F(70 + 580 * .8 / 2.8)}' y2='350' stroke='red' stroke-dasharray='5,4'/><text x='430' y='30' font-size='12'>T=1 (rouge), KL (bleu)</text>");
chart.Append("<polyline fill='none' stroke='blue' stroke-width='2' points='");
for (int i = 0; i < 29; i++)
    chart.Append($"{F(70 + 580 * i / 28.0)},{F(350 - 300 * ks[i] / km)} ");
chart.Append("'/><text x='270' y='410' font-size='14'>Température T</text><text x='80' y='30' font-size='14'>KL(ref || ref@T), bits</text></svg>");
display(new Svg(chart.ToString()));
foreach (double t in new[]
{
    .3,
    1d,
    2d
}

)
    if (Math.Abs(KL(new[] { .5, .5 }, Temp(new[] { .5, .5 }, t))) > 1e-12)
        throw new Exception("uniforme");
foreach (double t in new[]
{
    0d,
    -1d,
    double.NaN,
    double.PositiveInfinity
}

)
{
    try
    {
        Temp(new[] { .5, .5 }, t);
        throw new Exception("température invalide acceptée");
    }
    catch (ArgumentException)
    {
    }
}

Console.WriteLine("Températures invalides et référence uniforme : OK");
var cold = Temp(new[] { .9, .1 }, 1e-320);
if (cold[0] != 1 || cold[1] != 0)
    throw new Exception("température subnormale");
Console.WriteLine("Température subnormale : OK");


T=0.3, KL=1.7047 bits, H=3.2406


T=1.0, KL=-0.0000 bits, H=5.2835


T=2.0, KL=0.1424 bits, H=5.7206


Minimum à T=1.0


0.20 0.00 0.90 0.94 1.60 1.87 2.30 2.81 3.00 3.74 T=1 (rouge), KL (bleu) Température T KL(ref || ref@T), bits

Températures invalides et référence uniforme : OK


Température subnormale : OK


**Lecture.** Sur le même texte utilisé pour entraîner le bigramme, la perte vaut 4,009 bits par transition ; le gain apparié face à l'unigramme sur ces mêmes 1 306 positions vaut 0,522 bit. Ce résultat en entraînement ne mesure pas la généralisation. Pour le contexte espace, H vaut 3,2406, 5,2835 et 5,7206 bits à T=0,3, 1 et 2 ; KL(ref || ref@T) vaut 1,7047, 0 et 0,1424 bit. T=1 est un minimum, sans être nécessairement unique : une référence uniforme est inchangée à toute température.

## Exercice 3 — Température et KL

Calculez KL(ref || ref@T) en bits pour une référence strictement positive et normalisée et une température finie T>0. Tracez la courbe sur les 29 températures de 0,2 à 3 et vérifiez que T=1 est un minimum, sans imposer son unicité. Refusez vecteurs vides, non finis, non positifs ou de masse différente de 1 et températures invalides.

In [9]:
double? KlTemperature(double[] reference, double temperature)
{ /* TODO étudiant */
    return null;
}

bool VerifierTemperature(Func<double[], double, double?> f)
{
    foreach (var test in new[]
    {
        (new[] { .2, .8 }, 1d),
        (new[] { .2, .8 }, 2d),
        (new[] { .5, .5 }, .3),
        (new[] { .7, .2, .1 }, .4)
    }

    )
    {
        var(r, t) = test;
        var val = f(r, t);
        if (val is null)
            return false;
        var pow = r.Select(x => Math.Pow(x, 1 / t)).ToArray();
        double sum = pow.Sum();
        double attendu = r.Zip(pow, (x, z) => x * Math.Log2(x / (z / sum))).Sum();
        if (!double.IsFinite(val.Value) || Math.Abs(val.Value - attendu) > 1e-10)
            throw new Exception("KL incorrecte");
    }

    foreach (var t in new[]
    {
        (new[] { .5, .5 }, 0d),
        (new[] { .5, .5 }, double.NaN),
        (Array.Empty<double>(), 1d),
        (new[] { double.PositiveInfinity, 1d }, 1d),
        (new[] { .5, .5 }, -1d),
        (new[] { .5, .5 }, double.PositiveInfinity),
        (new[] { -.1, 1.1 }, 1d),
        (new[] { .2, .2 }, 1d)
    }

    )
    {
        try
        {
            f(t.Item1, t.Item2);
            throw new Exception("entrée invalide acceptée");
        }
        catch (ArgumentException)
        {
        }
    }

    return true;
}

Console.WriteLine(VerifierTemperature(KlTemperature) ? "Exercice 3 validé" : "Exercice 3 à compléter");


Exercice 3 à compléter


***

## 5. Les ponts : les mêmes quantités dans DPO et GRPO

Les deux grandeurs manipulées ici — la **cross-entropy** et la **divergence de KL** — sont les briques des losses de toute la série. Deux exemples précis, dans [PT_03](../../../GenAI/PostTraining/PT_03_dpo_direct_preference.ipynb) (DPO) et [PT_04](../../../GenAI/PostTraining/PT_04_grpo_deepseek_r1.ipynb) :

- **DPO (PT_03)** : la récompense implicite est $\beta \cdot \log \frac{\pi_\theta}{\pi_{ref}}$, et la loss est un `-log sigmoid` du **différentiel de log-rapports** entre la réponse choisie et la réponse rejetée. C'est un **log-ratio de probabilités** ($\log \pi_\theta - \log \pi_{ref}$) — la même différence de log-probas que notre $\log_2 \frac{p}{q}$, à la base près. Le paramètre $\beta = 0.1$ (PT_03) est le **coefficient de régularisation KL**.
- **GRPO/PPO (PT_04)** : la loss est un `clipped surrogate` $\min(r_i \hat{A}_i, \mathrm{clip}(r_i, 1-\varepsilon, 1+\varepsilon) \hat{A}_i)$ **plus un penalty KL** $\beta \cdot D_{KL}(\pi_\theta \| \pi_{ref})$. Le `ratio` $r_i = \pi_\theta / \pi_{old}$ est un rapport de probabilités, et le penalty KL est **exactement** notre $D_{KL}$ de la section 2. Le $\beta = 0.04$ (PT_04, défaut DeepSeek) règle la force de ce rappel vers la référence.

Un coefficient de rappel faible peut permettre à la politique de s'éloigner de sa référence, sans impliquer à lui seul une panne numérique ou un logarithme de zéro. En entraînement, cette KL théorique est souvent estimée sur des échantillons ; le calcul suivant en donne la somme exacte sur quatre tokens, sans reproduire un entraînement GRPO réel.


In [10]:
double[] Softmax(double[] z)
{
    double m = z.Max();
    return Norm(z.Select(x => Math.Exp(x - m)).ToArray());
}

var policy = Softmax(new[] { 3d, 1.5, 4, .5 });
var policyRef = Softmax(new[] { 2.8, 2.6, 3.2, 1.8 });
var ratios = policy.Zip(policyRef, (a, b) => Math.Log(a / b)).ToArray();
double dpo = Soft(-.1 * (ratios[2] - ratios[1]));
double bits = KL(policy, policyRef), nats = policy.Zip(ratios, (a, b) => a * b).Sum();
Console.WriteLine($"log-ratios naturels [{string.Join(", ", ratios.Select(x => x.ToString("F3")))}] ; récompenses beta=0,1 [{string.Join(", ", ratios.Select(x => (.1 * x).ToString("F3")))}]");
Console.WriteLine($"Perte DPO {dpo:F4} ; KL {bits:F4} bits = {nats:F4} nats");


log-ratios naturels [-0.090, -1.390, 0.510, -1.590] ; récompenses beta=0,1 [-0.009, -0.139, 0.051, -0.159]


Perte DPO 0.6027 ; KL 0.3073 bits = 0.2130 nats


**Lecture.** Le différentiel de log-rapports naturels donne une perte DPO jouet de 0,6027. La KL exacte sur quatre tokens vaut 0,2130 nat, soit 0,3073 bit : la conversion multiplie les bits par ln(2). Le calcul illustre les grandeurs utilisées par les méthodes de préférence sans reproduire leur entraînement sur des séquences.

## Contre-sens usuels et récapitulatif des exercices

La CE est égale à l'entropie plus la KL ; celle-ci est non négative et asymétrique. Un zéro sur le support de p donne une CE infinie, sauf si l'on change la distribution par un lissage normalisé. La demi-MSE peut apprendre, mais son gradient peut être faible en saturation. Le gain du bigramme est ici mesuré sur l'entraînement. Une température neutre donne KL=0, éventuellement avec d'autres minima pour une référence uniforme.

Exercices : [calculer l’entropie du second texte](#Exercice-1--Entropie-du-second-texte), [lisser une distribution et calculer sa CE](#Exercice-2--Lissage-normalisé), puis [étudier KL en fonction de la température](#Exercice-3--Température-et-KL). Les trois stubs ci-dessus restent à compléter.

Voir aussi le [glossaire](GLOSSAIRE.md).

## À retenir

L'entropie mesure une surprise moyenne ; la CE ajoute le surcoût KL d'un modèle. Le cas saturé illustre une différence de gradients, et le modèle de caractères illustre une comparaison sur l'entraînement.

## Vérifiez votre compréhension

Pourquoi une KL n'est-elle pas une distance ?

<details><summary>Réponse</summary>Elle est asymétrique.</details>

Pourquoi T=1 peut-il ne pas être le seul minimum ?

<details><summary>Réponse</summary>Une distribution uniforme reste uniforme pour toute température positive.</details>

## Pour aller plus loin

Les ouvrages suivants développent les fondements probabilistes et les liens avec l’apprentissage.

## Références

1. Shannon, C. E. (1948). *A Mathematical Theory of Communication*. Bell System Technical Journal, 27, 379–423.
2. Kullback, S. & Leibler, R. A. (1951). *On Information and Sufficiency*. Annals of Mathematical Statistics, 22(1), 79–86.
3. Goodfellow, I., Bengio, Y. & Courville, A. (2016). *Deep Learning*, section 3.13, p. 71. MIT Press. https://www.deeplearningbook.org/contents/TOC.html
